In [ ]:
# Install required packages
pip install python-jose cryptography 2>&1 | tail -5 install -q langchain-core langchain-community langchain-openai langchain-text-splitters langchain-experimental faiss-cpu python-dotenv pyyaml numpy pandas pypdf PyMuPDF rank-bm25

# Final Evaluation – A0 vs A7 vs A8

Reads all experiment JSONL results and produces summary tables for Chapter 4.

In [ ]:
import sys, json
import numpy as np, pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
from report_common.io import load_jsonl, save_csv_summary

## Load All Results

In [ ]:
results_dir = PROJECT_ROOT / 'report_results'

experiments = {}
for jsonl_file in sorted(results_dir.glob('A*.jsonl')):
    records = load_jsonl(jsonl_file)
    exp_name = jsonl_file.stem
    experiments[exp_name] = records
    print(f'  {exp_name}: {len(records)} records')

print(f'\nLoaded {len(experiments)} experiment files.')

## Aggregate Metrics per Experiment

In [ ]:
summary_rows = []

for exp_name, records in experiments.items():
    row = {'experiment': exp_name, 'n_questions': len(records)}

    # Numeric metrics
    metric_keys = [k for k in records[0].get('metrics', {}) if isinstance(records[0]['metrics'].get(k), (int, float))]
    for key in metric_keys:
        vals = [r['metrics'][key] for r in records if isinstance(r['metrics'].get(key), (int, float))]
        if vals:
            row[f'mean_{key}'] = round(np.mean(vals), 3)

    # Latency
    latencies = [r['latency'].get('total_seconds', 0) for r in records if 'latency' in r]
    if latencies:
        row['mean_latency_s'] = round(np.mean(latencies), 3)
        row['p95_latency_s'] = round(np.percentile(latencies, 95), 3)

    summary_rows.append(row)

df = pd.DataFrame(summary_rows)
print(df.to_string(index=False))

## Naive vs Advanced Comparison

In [ ]:
# Filter A0 and A7
naive = [r for r in summary_rows if 'A0' in r['experiment']]
advanced = [r for r in summary_rows if 'A7' in r['experiment']]

if naive and advanced:
    print('\n' + '='*60)
    print('NAIVE RAG (A0) vs ADVANCED RAG (A7)')
    print('='*60)
    n, a = naive[0], advanced[0]
    common_keys = [k for k in n if k in a and k.startswith('mean_')]
    for key in common_keys:
        nv, av = n.get(key, 0), a.get(key, 0)
        delta = av - nv if nv and av else None
        print(f'  {key:30s}: {nv} -> {av}  (delta={delta})')
    print('='*60)
else:
    print('Run A0 and A7 notebooks first to compare.')

## Results by Question Category

In [ ]:
# Load questions to get categories
with open(PROJECT_ROOT / 'report_data/evaluation/questions.json', 'r') as f:
    questions = json.load(f)
qid_to_cat = {q['question_id']: q['category'] for q in questions}

# Breakdown for A7
if 'A7_advanced_rag' in experiments:
    a7_records = experiments['A7_advanced_rag']
    cat_results = {}
    for r in a7_records:
        cat = qid_to_cat.get(r['question_id'], 'unknown')
        if cat not in cat_results: cat_results[cat] = []
        cat_results[cat].append(r)

    print('\nA7 Results by Category:')
    for cat, recs in sorted(cat_results.items()):
        abstain_correct = np.mean([r['metrics'].get('correct_abstain', 0) for r in recs])
        print(f'  {cat:20s}: n={len(recs)} abstain_acc={abstain_correct:.2f}')

## Save Summary CSV

In [ ]:
output_dir = results_dir / 'summary'
output_dir.mkdir(parents=True, exist_ok=True)
save_csv_summary(summary_rows, output_dir / 'all_experiments.csv')
print('Summary saved to report_results/summary/all_experiments.csv')

## Failure Cases

In [ ]:
# Show cases where A7 still failed
if 'A7_advanced_rag' in experiments:
    failures = [r for r in experiments['A7_advanced_rag']
                if not r['metrics'].get('correct_abstain', True)]
    print(f'\nA7 Failure cases: {len(failures)}')
    for f in failures[:5]:
        print(f'  [{f["question_id"]}] predicted_abstain={f["predicted_abstain"]} | {f["question"][:60]}')